<div dir="rtl">

# 🎓 أساسيات YOLO الجزء 1: المهام والاستدلال (Inference) 🚀

مرحباً بك في المعمل الأول لدورة Ultralytics YOLO! في هذا المعمل، سنركز حصرياً على تشغيل **الاستدلال** (وضع الـ `predict`) عبر مختلف المهام التي يدعمها YOLO.

لن تحتاج لتدريب أي نماذج اليوم؛ بدلاً من ذلك، ستستخدم أوزان `.pt` مدربة مسبقاً تأتي مباشرة من Ultralytics لترى ما يمكن لـ YOLO فعله بمجرد تشغيله.

</div>

<div dir="rtl">

## 🛠️ 1. تجهيز بيئة العمل
أولاً، لنقم بتثبيت مكتبة `ultralytics` والتأكد من جاهزية النظام.

</div>

In [ ]:
import sys
# التثبيت التلقائي للمكتبة إذا لم تكن موجودة
if 'google.colab' in sys.modules:
    %pip install -q ultralytics
else:
    !pip install -q ultralytics

import ultralytics
ultralytics.checks()

<div dir="rtl">

## 💻 2. أساسيات واجهة سطر الأوامر (CLI)

واجهة سطر الأوامر هي أسرع طريقة للبدء. الصيغة الأساسية هي: `yolo TASK MODE ARGS`.

لنقم بتشغيل مهمة **اكتشاف الكائنات** (Object Detection) باستخدام وضع التوقع (`predict`).

</div>

In [ ]:
# تشغيل YOLO من سطر الأوامر لتحليل صورة حافلة
!yolo task=detect mode=predict model=yolo26n.pt source="https://ultralytics.com/images/bus.jpg"

<div dir="rtl">

## 🐍 3. برمجة بايثون واستكشاف المهام

بينما تعتبر واجهة سطر الأوامر رائعة للاختبارات السريعة، فإن مكتبة بايثون (Python API) هي ما ستستخدمه لبناء تطبيقات حقيقية.

سنستكشف أدناه المهام الخمس الأساسية: **الاكتشاف، التصنيف، التجزئة، الوضعية، والمربعات المائلة**.

</div>

<div dir="rtl">

### 🎯 3.1 اكتشاف الكائنات (Object Detection)
تجد الكائنات وتحدد موقعها باستخدام صناديق الإحاطة (Bounding Boxes).

</div>

In [ ]:
from ultralytics import YOLO
from PIL import Image

# 1. تحميل نموذج اكتشاف مدرب مسبقاً
model = YOLO("yolo26n.pt")

# 2. تشغيل التوقع
results = model.predict(source="https://ultralytics.com/images/zidane.jpg")

# 3. استكشاف النتائج
result = results[0]
print(f"تم اكتشاف {len(result.boxes)} كائنات.")
print(f"شكل مصفوفة الصناديق (N, 6): {result.boxes.data.shape}")

# عرض الصورة مع الرسم التوضيحي
Image.fromarray(result.plot()[:,:,::-1]) # تحويل من BGR إلى RGB للعرض الصحيح

<div dir="rtl">

### 💡 تمرين: استخراج الإحداثيات يدوياً

بدلاً من الاعتماد على `.plot()`، قم باستخراج الإحداثيات الخام لـ **أول كائن مكتشف** واطبع قيم $(x1, y1, x2, y2)$.

*تلميح: تحقق من result.boxes[0].xyxy*

</div>

In [ ]:
# TODO: استخراج وطباعة الإحداثيات (x1, y1, x2, y2) للكائن الأول


<div dir="rtl">

<details>
<summary>عرض الحل 💡</summary>

```python
first_box = result.boxes[0]
coords = first_box.xyxy[0].tolist()
print(f"الإحداثيات: {coords}")
```

</details>

</div>

<div dir="rtl">

### 🏷️ 3.2 تصنيف الصور (Image Classification)
تقوم بإعطاء تسمية واحدة للصورة بأكملها.

</div>

In [ ]:
model_cls = YOLO("yolo26n-cls.pt")
results_cls = model_cls.predict(source="https://ultralytics.com/images/bus.jpg")

result = results_cls[0]
print(f"رقم الفئة الأعلى (Top-1 index): {result.probs.top1}")
print(f"اسم الفئة الأعلى (Top-1 label): {result.names[result.probs.top1]}")
print(f"نسبة الثقة (Top-1 Confidence): {result.probs.top1conf:.2f}")

Image.fromarray(result.plot()[:,:,::-1])

<div dir="rtl">

### 💡 تمرين: تصفية النتائج بناءً على الثقة

قم بتصفية مصفوفة الاحتمالات (`probs.data`) يدوياً. اطبع أرقام الفئات التي تزيد نسبة الثقة فيها عن **0.05**.

</div>

In [ ]:
# TODO: ابحث واطبع أرقام الفئات التي احتمالها > 0.05


<div dir="rtl">

<details>
<summary>عرض الحل 💡</summary>

```python
import numpy as np
indices = np.argwhere(result.probs.data > 0.05)[0]
print(f"الفئات ذات الثقة العالية: {indices.tolist()}")
```

</details>

</div>

<div dir="rtl">

### ✂️ 3.3 التجزئة (Instance Segmentation)
تحدد الشكل الدقيق (القناع) لكل كائن بكسل ببكسل.

</div>

In [ ]:
model_seg = YOLO("yolo26n-seg.pt")
results_seg = model_seg.predict(source="https://ultralytics.com/images/zidane.jpg")

result = results_seg[0]
if result.masks:
    print(f"شكل مصفوفة الأقنعة (N, H, W): {result.masks.data.shape}")

Image.fromarray(result.plot()[:,:,::-1])

<div dir="rtl">

### 🤸 3.4 تقدير الوضعية (Pose Estimation)
تكتشف الأشخاص وتبرز النقاط التشريحية الرئيسية (المفاصل).

</div>

In [ ]:
model_pose = YOLO("yolo26n-pose.pt")
results_pose = model_pose.predict(source="https://ultralytics.com/images/bus.jpg")

result = results_pose[0]
if result.keypoints:
    print(f"شكل مصفوفة النقاط (الأشخاص، النقاط، الإحداثيات): {result.keypoints.data.shape}")

Image.fromarray(result.plot()[:,:,::-1])

<div dir="rtl">

### 📐 3.5 المربعات المائلة (Oriented Bounding Boxes - OBB)
تكتشف الكائنات باستخدام صناديق مائلة، وهي مثالية للتصوير الجوي أو من الأعلى.

</div>

In [ ]:
# تستخدم Ultralytics أفضل نموذج متاح إذا كان الموديل عاماً
model_obb = YOLO("yolo11n-obb.pt") 

# استخدام صورة جوية لتجربة المربعات المائلة
results_obb = model_obb.predict(source="https://ultralytics.com/images/boats.jpg")

result = results_obb[0]
if getattr(result, "obb", None) is not None:
    print(f"شكل مصفوفة OBB (الصناديق، 7): {result.obb.data.shape}")
    print("خصائص OBB: المركز x, المركز y, العرض, الارتفاع, الزاوية, الثقة, الفئة")

Image.fromarray(result.plot()[:,:,::-1])

<div dir="rtl">

---
## 🏁 نهاية المعمل
لقد نجحت في تشغيل الاستدلال (وضع الـ `predict`) عبر جميع مهام YOLO الخمس الرئيسية باستخدام واجهة سطر الأوامر ومكتبة بايثون!

في الجزء الثاني من هذه الدورة، سننتقل للتعامل مع البيانات المخصصة والتدريب (وضع الـ `train`).

</div>